In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import gc
import os
import sys
import ctypes
from tqdm import tqdm

warnings.filterwarnings('ignore')
# Vypnutie NNPACK akcelerácie - na niektorých CPU hádže nezmyselné warningy
os.environ['NNPACK_DISABLE'] = '1'
# Nepoužívať CUDA memory caching - šetrí GPU RAM (potrebné pri menších GPU)
os.environ['PYTORCH_NO_CUDA_MEMORY_CACHING'] = '1'

# Potlačenie C++ NNPACK warningov
import logging
logging.disable(logging.WARNING)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Potlač všetky PyTorch warningy
torch.set_warn_always(False)
logging.disable(logging.NOTSET)
logging.getLogger('torch').setLevel(logging.CRITICAL)

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, f1_score,
                             precision_score, recall_score,
                             roc_auc_score, average_precision_score)

# Ak je dostupná GPU, použijeme ju (TCN má dosť parametrov, na CPU by bežala dlho)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Používam: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ═══════════════════════════════════════════════════
# KONFIGURÁCIA
# ═══════════════════════════════════════════════════
DATA_PATH = '../Data/Export_data/forecast_30min.parquet'
OUTPUT_DIR = '../Modely/TemporalCNN/tcnn_30min'
CACHE_DIR = '../Modely/TemporalCNN/tcnn_30min/cache'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

VOLTAGE_COLS = ['u1_norm', 'u2_norm', 'u3_norm']
CURRENT_COLS = ['i1_norm', 'i2_norm', 'i3_norm']
TARGET_COLS = VOLTAGE_COLS + CURRENT_COLS

SELECTED_HORIZONS = [2, 6, 12, 24]

# Vstupná sekvencia = koľko krokov histórie model vidí pri predikcii
SEQUENCE_LENGTH = 336
FORECAST_HORIZON = 24
MIN_SEGMENT_LENGTH = SEQUENCE_LENGTH + FORECAST_HORIZON

# Architektúra TCN - 3 vrstvy postupne s 32, 64, 64 kanálmi
# Každá ďalšia vrstva má vyšší dilation faktor (1, 2, 4) - tak vidí ďalej do minulosti
NUM_CHANNELS = [32, 64, 64]
KERNEL_SIZE = 3
DROPOUT = 0.2

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 5  # zastaví, ak sa val loss 5 epoch nezlepší

ANOMALY_SIGMA = 3.0

print(f"Sekvencia: {SEQUENCE_LENGTH}, Horizont: {FORECAST_HORIZON}")
print(f"Horizonty: {SELECTED_HORIZONS}")


---
# ČASŤ 1: NAČÍTANIE DÁT A ROZDELENIE
---

In [ ]:
%%time
data = pd.read_parquet(DATA_PATH)
data['t_utc'] = pd.to_datetime(data['t_utc'])
# float32 namiesto float64 šetrí pamäť na polovicu
for col in data.select_dtypes(include=['float64']).columns:
    data[col] = data[col].astype('float32')

print(f"Celkovo: {len(data):,} riadkov")
print(f"Chyba=0: {(data['Chyba']==0).sum():,}, Chyba=1: {(data['Chyba']==1).sum():,}")

# Pre každý segment zistíme jeho stav (chyba_max=0 znamená, že celý segment je čistý)
seg_info = data.groupby(['eic', 'segment_id'], observed=True).agg(
    start_time=('t_utc', 'min'), end_time=('t_utc', 'max'),
    num_records=('t_utc', 'count'), chyba_max=('Chyba', 'max')).reset_index()

# Rozdelenie na čisté a poruchové segmenty - sortujeme podľa času pre chronologický split
clean_segs = seg_info[seg_info['chyba_max'] == 0].sort_values('start_time').reset_index(drop=True)
error_segs = seg_info[seg_info['chyba_max'] == 1].sort_values('start_time').reset_index(drop=True)

# Chronologický split čistých segmentov 70/15/15 (train/val/test)
n_clean = len(clean_segs)
cut_70 = int(n_clean * 0.70)
cut_85 = int(n_clean * 0.85)

# Sety dvojíc (eic, segment_id) - rýchly lookup pri filtrovaní dát nižšie
train_seg_keys = set(zip(clean_segs.iloc[:cut_70]['eic'], clean_segs.iloc[:cut_70]['segment_id']))
val_seg_keys = set(zip(clean_segs.iloc[cut_70:cut_85]['eic'], clean_segs.iloc[cut_70:cut_85]['segment_id']))
test_seg_keys = set(zip(clean_segs.iloc[cut_85:]['eic'], clean_segs.iloc[cut_85:]['segment_id']))
error_seg_keys = set(zip(error_segs['eic'], error_segs['segment_id']))

# Filtrovanie dát podľa setov
seg_key = list(zip(data['eic'], data['segment_id']))
train_clean = data[np.array([k in train_seg_keys for k in seg_key])].copy()
val_clean = data[np.array([k in val_seg_keys for k in seg_key])].copy()
test_clean = data[np.array([k in test_seg_keys for k in seg_key])].copy()
error_data = data[np.array([k in error_seg_keys for k in seg_key])].copy()
# anomaly_data = test_clean (Chyba=0) + error_data (Chyba=1) - na test anomaly detection
anomaly_data = pd.concat([test_clean, error_data], ignore_index=True)

del seg_key, data; gc.collect()

# Súhrnná tabuľka rozdelenia
sep = '=' * 70
print(f"\n{sep}")
print(f"{'MNOŽINA':<22s} {'RIADKOV':>12s} {'Chyba=0':>12s} {'Chyba=1':>12s} {'Segmentov':>10s}")
print(sep)
for nm, df in [('train_clean', train_clean), ('val_clean', val_clean),
               ('test_clean', test_clean), ('error_data', error_data), ('anomaly_data', anomaly_data)]:
    c0 = (df['Chyba']==0).sum(); c1 = (df['Chyba']==1).sum()
    n_seg = df.groupby(['eic','segment_id'], observed=True).ngroups
    print(f"  {nm:<20s} {len(df):>10,} {c0:>10,} {c1:>10,} {n_seg:>9,}")
print(sep)


---
# ČASŤ 2: DATASET A DATALOADER
---

In [ ]:
class TimeSeriesDataset(Dataset):
    """Memory-efficient dataset - ukladá numpy, tensory vytvára on-the-fly."""
    # Kľúčový rozdiel oproti naivnému prístupu: x_data a y_data sú numpy arrays.
    # Tensory sa vytvárajú až v __getitem__ - pri miliónoch sekvencií by inak RAM nestíhala.
    def __init__(self, df, seq_len, horizon, target_cols, scaler=None, fit_scaler=False, step=1):
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_cols = target_cols
        
        # Ukladáme len indexy a dáta ako numpy (nie tensory)
        self.x_data = []
        self.y_data = []
        self.meta = []  # (eic, seg_id, times, chyba) - na neskoršie spárovanie predikcií

        # Scaler fitujeme len na train datasete, val/test ho dostanú už natrénovaný
        # RobustScaler je odolný voči outlierom (škáluje podľa mediánu a IQR)
        if fit_scaler:
            self.scaler = RobustScaler()
            self.scaler.fit(df[target_cols])
        else:
            self.scaler = scaler

        # Pre každý segment vytvoríme sliding window sekvencie
        segs = df.groupby(['eic', 'segment_id'], observed=True)
        for (eic, seg_id), seg_data in tqdm(segs, desc="Príprava dát"):
            seg_data = seg_data.sort_values('t_utc')
            if len(seg_data) < seq_len + horizon:
                continue
            values = seg_data[target_cols].values.astype('float32')
            # Škálovanie - TCN funguje stabilnejšie na škálovaných dátach
            values_scaled = self.scaler.transform(values).astype('float32')
            times = seg_data['t_utc'].values
            chyba = seg_data['Chyba'].values

            # Sliding window: pre každú možnú pozíciu vyrobíme dvojicu (X, y).
            # step=1 pri tréningu (max samples), step=12 pri anomaly detection (rýchlejšie)
            for i in range(0, len(values_scaled) - seq_len - horizon + 1, step):
                self.x_data.append(values_scaled[i:i+seq_len])              # SEQ_LEN krokov histórie
                self.y_data.append(values_scaled[i+seq_len:i+seq_len+horizon])  # HORIZON krokov budúcnosti
                self.meta.append((eic, seg_id, 
                                  times[i+seq_len:i+seq_len+horizon],
                                  chyba[i+seq_len:i+seq_len+horizon]))
        
        print(f"  Vytvorených {len(self.x_data):,} samples (step={step})")

    def __len__(self): return len(self.x_data)
    
    def __getitem__(self, idx):
        # Tensory sa vytvárajú až tu - on-the-fly pri každom batch-i
        return torch.from_numpy(self.x_data[idx]), torch.from_numpy(self.y_data[idx])


In [ ]:
%%time
# Train dataset - tu fitujeme scaler (zistí mediány/IQR z trénovacích dát)
print("Príprava TRAIN datasetu (Chyba=0)...")
train_dataset = TimeSeriesDataset(train_clean, SEQUENCE_LENGTH, FORECAST_HORIZON,
                                   TARGET_COLS, fit_scaler=True, step=1)
scaler = train_dataset.scaler
del train_clean; gc.collect()

# Val a test používajú scaler z trainu (žiadne data leakage)
print("\nPríprava VAL datasetu (Chyba=0)...")
val_dataset = TimeSeriesDataset(val_clean, SEQUENCE_LENGTH, FORECAST_HORIZON,
                                 TARGET_COLS, scaler=scaler, step=1)
del val_clean; gc.collect()

print("\nPríprava TEST datasetu (Chyba=0)...")
test_dataset = TimeSeriesDataset(test_clean, SEQUENCE_LENGTH, FORECAST_HORIZON,
                                  TARGET_COLS, scaler=scaler, step=1)
del test_clean; gc.collect()

# DataLoader - posúva batch-e do modelu počas tréningu.
# num_workers=0 a pin_memory=False - pre stabilitu (multi-worker hádzal občas chyby)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f"\nTrain: {len(train_dataset):,}, Val: {len(val_dataset):,}, Test: {len(test_dataset):,}")


In [ ]:
# Temporal CNN architektúra (Bai et al., 2018) - kombinácia troch stavebných blokov

class CausalConv1d(nn.Module):
    # Causal konvolúcia = pri predikcii kroku t používa len kroky <= t (nie budúcnosť).
    # Dosahuje sa to padding-om vľavo a oddelením pravej časti výstupu.
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        # Padding pre causal: (kernel_size-1)*dilation - tak veľa zľava, aby výstup bol rovnaký
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=self.padding, dilation=dilation)
    def forward(self, x):
        x = self.conv(x)
        # Odrežeme posledných `padding` krokov - tým zabránime, aby konvolúcia "videla budúcnosť"
        if self.padding > 0: x = x[:, :, :-self.padding]
        return x

class TemporalBlock(nn.Module):
    # Stavebný blok TCN: dve causal konvolúcie + BatchNorm + ReLU + Dropout + residual connection.
    # Residual connection (in → out) zlepšuje gradient flow pri hlbších sieťach
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        # Ak sa líši počet kanálov vstupu a výstupu, downsample (1x1 konvolúcia) ich zladí
        # aby sme mohli urobiť residual sum (potrebujú rovnaký shape)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        residual = x
        out = self.dropout(self.relu(self.bn1(self.conv1(x))))
        out = self.dropout(self.relu(self.bn2(self.conv2(out))))
        if self.downsample is not None: residual = self.downsample(residual)
        return self.relu(out + residual)

class TemporalCNN(nn.Module):
    # Hlavný model: zreťazené TemporalBlocky + finálna FC vrstva pre predikciu.
    # Dilation rastie exponenciálne (1, 2, 4, ...) - každá ďalšia vrstva vidí 2x ďalej do minulosti
    def __init__(self, input_size, num_channels, kernel_size, dropout, forecast_horizon):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation = 2 ** i  # exponenciálne rastúci dilation
            in_ch = input_size if i == 0 else num_channels[i-1]
            layers.append(TemporalBlock(in_ch, num_channels[i], kernel_size, dilation, dropout))
        self.network = nn.Sequential(*layers)
        # FC vrstva mapuje posledný hidden state na forecast - výstup má (horizon × features) hodnôt
        self.fc = nn.Linear(num_channels[-1], input_size * forecast_horizon)
        self.input_size = input_size
        self.forecast_horizon = forecast_horizon

    def forward(self, x):
        # PyTorch Conv1d očakáva (batch, channels, time) ale dataset dáva (batch, time, features)
        # transpose prehodí dve posledné dimenzie
        x = x.transpose(1, 2)
        x = self.network(x)
        # Vezmeme len posledný časový krok (obsahuje informáciu z celej sekvencie)
        x = x[:, :, -1]
        x = self.fc(x)
        # Reshape: (batch, horizon * features) -> (batch, horizon, features)
        return x.view(-1, self.forecast_horizon, self.input_size)

# Vytvorenie modelu, loss funkcie a optimizéra
model = TemporalCNN(
    input_size=len(TARGET_COLS), num_channels=NUM_CHANNELS,
    kernel_size=KERNEL_SIZE, dropout=DROPOUT,
    forecast_horizon=FORECAST_HORIZON).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# Scheduler zníži LR na polovicu, ak sa val loss 3 epochy nezlepší
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"Parametrov: {sum(p.numel() for p in model.parameters()):,}")


---
# ČASŤ 4: TRÉNOVANIE
---

In [ ]:
%%time
def train_epoch(model, loader, criterion, optimizer, device):
    # Jedna epocha tréningu - prejde všetky batch-e raz
    model.train()
    total_loss = 0
    for x, y in tqdm(loader, desc='Train', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        # Gradient clipping - zabráni exploding gradients (typický problém pri RNN/CNN sieťach)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, device):
    # Evaluation - bez gradientov (no_grad šetrí pamäť a urýchľuje)
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total_loss += criterion(model(x), y).item()
    return total_loss / len(loader)

print("=" * 60)
print("TEMPORAL CNN - TRÉNING (len Chyba=0)")
print("=" * 60)

# Tréningová slučka s early stopping a uložením najlepšieho modelu
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = eval_epoch(model, val_loader, criterion, device)
    train_losses.append(train_loss); val_losses.append(val_loss)
    scheduler.step(val_loss)

    # Ak je val loss najlepšia doteraz, uložíme model na disk a resetujeme patience
    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_counter = 0
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/tcnn_best.pt')
        status = ' ✓'
    else:
        patience_counter += 1; status = ''

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}{status}")

    # Early stopping - ak sa nezlepšila val loss EARLY_STOPPING_PATIENCE epoch, končíme
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping po {epoch+1} epochách"); break

print(f"\nNajlepšia val loss: {best_val_loss:.6f}")

del train_loader, val_loader, train_dataset, val_dataset
gc.collect()


---
# ČASŤ 5: EVALUACIA NA TEST SETE
---

In [ ]:
# Načítame najlepší model (uložený počas tréningu pri najnižšej val loss)
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/tcnn_best.pt', map_location=device))
model.eval()

def evaluate_dataset_batch(model, dataset, scaler, device, split_name, batch_size=64):
    """Batch evaluácia - rýchlejšia a menej pamäťovo náročná."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    
    all_preds = []
    all_trues = []
    
    # Forward pass cez všetky batch-e a zber predikcií
    model.eval()
    with torch.no_grad():
        for x, y in tqdm(loader, desc=f'Eval {split_name}'):
            x = x.to(device)
            pred = model(x).cpu().numpy()
            all_preds.append(pred)
            all_trues.append(y.numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)
    
    # Inverse transform - dáta boli škálované, teraz ich vrátime na pôvodnú škálu (V, A)
    n_samples = len(all_preds)
    horizon = all_preds.shape[1]
    n_features = all_preds.shape[2]
    
    # scaler očakáva 2D array, takže najprv reshape, potom inverse transform, potom späť na 3D
    preds_flat = all_preds.reshape(-1, n_features)
    trues_flat = all_trues.reshape(-1, n_features)
    preds_orig = scaler.inverse_transform(preds_flat).reshape(n_samples, horizon, n_features)
    trues_orig = scaler.inverse_transform(trues_flat).reshape(n_samples, horizon, n_features)
    
    # Zostavenie results DataFrame - rozbalíme batch dimenziu na individuálne riadky
    results = []
    target_cols = dataset.target_cols
    for i in range(n_samples):
        meta = dataset.meta[i]
        eic, seg_id, times, chyba = meta
        for h_idx in range(horizon):
            h = h_idx + 1
            # Uložíme len vybrané horizonty (napr. h=1, 3, 6, 12)
            if h not in SELECTED_HORIZONS: continue
            for j, col in enumerate(target_cols):
                results.append({
                    'eic': eic, 'segment_id': seg_id,
                    't_utc': times[h_idx], 'Chyba': int(chyba[h_idx]),
                    'target': col, 'hour_ahead': h,
                    'prediction': float(preds_orig[i, h_idx, j]),
                    'actual': float(trues_orig[i, h_idx, j]),
                    'split': split_name})
    
    del all_preds, all_trues, preds_flat, trues_flat, preds_orig, trues_orig
    gc.collect()
    return pd.DataFrame(results)

print("Evaluácia na čistých dátach...")
forecast_df = evaluate_dataset_batch(model, test_dataset, scaler, device, 'test')
forecast_df['residual'] = forecast_df['actual'] - forecast_df['prediction']
forecast_df['abs_error'] = np.abs(forecast_df['residual'])

del test_dataset; gc.collect()
print(f"Predikcií (čisté): {len(forecast_df):,}")


In [ ]:
# Forecasting metriky pre test set, oddelene pre napätie a prúd
print(f"\n{'='*60}")
print(f"  TEST (Chyba=0): {len(forecast_df):,} predikcií")
print(f"{'='*60}")

print(f"\n  NAPÄTIE:")
rows_v = []
for t in VOLTAGE_COLS:
    d = forecast_df[forecast_df['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    rows_v.append({'target': t, 'MAE': mean_absolute_error(d['actual'], d['prediction']),
        'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])), 'R2': r2_score(d['actual'], d['prediction'])})
if rows_v:
    mv = pd.DataFrame(rows_v); print(mv.to_string(index=False))
    print(f"    → Priemer: MAE={mv['MAE'].mean():.4f}, RMSE={mv['RMSE'].mean():.4f}, R²={mv['R2'].mean():.4f}")

print(f"\n  PRÚDY:")
rows_c = []
for t in CURRENT_COLS:
    d = forecast_df[forecast_df['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    rows_c.append({'target': t, 'MAE': mean_absolute_error(d['actual'], d['prediction']),
        'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])), 'R2': r2_score(d['actual'], d['prediction'])})
if rows_c:
    mc = pd.DataFrame(rows_c); print(mc.to_string(index=False))
    print(f"    → Priemer: MAE={mc['MAE'].mean():.4f}, RMSE={mc['RMSE'].mean():.4f}, R²={mc['R2'].mean():.4f}")


In [ ]:
# Ako klesá presnosť so vzdialenosťou predikcie
print("METRIKY PODĽA HORIZONTU (test, Chyba=0):")

print("\n  Napätie:")
for h in SELECTED_HORIZONS:
    d = forecast_df[(forecast_df['hour_ahead']==h) & (forecast_df['target'].isin(VOLTAGE_COLS))]
    if len(d) < 2: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")

print("\n  Prúdy:")
for h in SELECTED_HORIZONS:
    d = forecast_df[(forecast_df['hour_ahead']==h) & (forecast_df['target'].isin(CURRENT_COLS))]
    if len(d) < 2: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")


In [ ]:
# Learning curve - ukáže priebeh tréningu (či model konverguje, či netrpí na overfitting).
# Ideál: train aj val loss klesajú podobne; ak val rastie pri klesajúcom train, je to overfitting
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='Train loss', markersize=4)
ax.plot(range(1, len(val_losses)+1), val_losses, 'r-s', label='Val loss', markersize=4)
ax.set_xlabel('Epocha'); ax.set_ylabel('Loss (MSE)')
ax.set_title('Learning Curve – Temporal CNN'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/learning_curve.png', dpi=150, bbox_inches='tight'); plt.show()


---
# ČASŤ 6: ANOMALY DETECTION
---

In [ ]:
%%time
print("=" * 60)
print("ANOMALY DETECTION (TCNN)")
print("=" * 60)

# Anomaly dataset má step=12 namiesto 1 - tým 12x menej samples a rýchlejšia evaluácia.
# Pri anomaly detection nepotrebujeme každý možný okno, postačí každé 12-te.
print("\nPríprava ANOMALY datasetu (step=12)...")
anomaly_dataset = TimeSeriesDataset(anomaly_data, SEQUENCE_LENGTH, FORECAST_HORIZON,
                                     TARGET_COLS, scaler=scaler, step=12)
del anomaly_data, error_data; gc.collect()

print(f"Anomaly samples: {len(anomaly_dataset):,}")

# Spustíme model na anomaly dáta - väčší batch_size (128) lebo netrénujeme
anomaly_df = evaluate_dataset_batch(model, anomaly_dataset, scaler, device, 'anomaly', batch_size=128)
anomaly_df['abs_error'] = np.abs(anomaly_df['actual'] - anomaly_df['prediction'])

del anomaly_dataset; gc.collect()

# Anomaly score = priemerná |chyba| cez 6 targetov v danom čase.
# Vyšší score = predikcia sa viac líši od skutočnosti = pravdepodobnejšia anomália
anom_agg = anomaly_df.groupby(['eic', 'segment_id', 't_utc', 'hour_ahead'], observed=True).agg(
    Chyba=('Chyba', 'first'), anomaly_score=('abs_error', 'mean')).reset_index()

del anomaly_df; gc.collect()

print(f"\nanom_agg: {len(anom_agg):,} (Chyba=0: {(anom_agg['Chyba']==0).sum():,}, Chyba=1: {(anom_agg['Chyba']==1).sum():,})")

# Prah z čistých dát - mean + 3σ z reziduí na normálnych dátach (baseline správanie modelu)
clean_scores = anom_agg[anom_agg['Chyba'] == 0]['anomaly_score'].values
threshold = clean_scores.mean() + ANOMALY_SIGMA * clean_scores.std()
print(f"\nPrah (mean + {ANOMALY_SIGMA}σ): {threshold:.4f}")

anom_agg['predicted_anomaly'] = (anom_agg['anomaly_score'] > threshold).astype(int)
y_true = anom_agg['Chyba'].values; y_pred = anom_agg['predicted_anomaly'].values
scores = anom_agg['anomaly_score'].values

# Confusion matrix - .ravel() rozbalí maticu 2x2 na 4 čísla [tn, fp, fn, tp]
cm = confusion_matrix(y_true, y_pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()

print(f"\nCONFUSION MATRIX:")
print(f"                  Predikované")
print(f"                Normal  Anomália")
print(f"  Skut. Normal  {tn:>7,}  {fp:>7,}")
print(f"  Skut. Chyba   {fn:>7,}  {tp:>7,}")
print(f"\n  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
# ROC-AUC a PR-AUC nepotrebujú prah - pracujú priamo s anomaly_score
try:
    print(f"  ROC-AUC:   {roc_auc_score(y_true, scores):.4f}")
    print(f"  PR-AUC:    {average_precision_score(y_true, scores):.4f}")
except: pass

# Metriky podľa horizontu
print(f"\nPODĽA HORIZONTU:")
for h in SELECTED_HORIZONS:
    hd = anom_agg[anom_agg['hour_ahead']==h]
    if len(hd) == 0: continue
    print(f"  h{h:2d}: P={precision_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}, "
          f"R={recall_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}, "
          f"F1={f1_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}")


In [ ]:
# Citlivostná analýza prahu - skúsime rôzne σ a pozrieme, ako sa menia precision/recall.
# Pomáha vidieť trade-off: nižšie σ = viac true positives ale aj viac false positives.
print("THRESHOLD ANALÝZA:")
print(f"{'σ':>5s} {'Prec':>8s} {'Recall':>8s} {'F1':>8s} {'TP':>7s} {'FP':>7s} {'FN':>7s}")
sigma_results = []
for sigma in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    th = clean_scores.mean() + sigma * clean_scores.std()
    yp = (scores > th).astype(int)
    sp = precision_score(y_true, yp, zero_division=0)
    sr = recall_score(y_true, yp, zero_division=0)
    sf = f1_score(y_true, yp, zero_division=0)
    s_cm = confusion_matrix(y_true, yp, labels=[0,1]); _, sfp, sfn, stp = s_cm.ravel()
    sigma_results.append({'σ': sigma, 'P': sp, 'R': sr, 'F1': sf})
    print(f"{sigma:5.1f} {sp:8.4f} {sr:8.4f} {sf:8.4f} {stp:7d} {sfp:7d} {sfn:7d}")

# Vykreslenie ako funkcie σ - vidíme, kde sa P, R, F1 "prelomia"
sr_df = pd.DataFrame(sigma_results)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sr_df['σ'], sr_df['P'], 'b-o', label='Precision', linewidth=2)
ax.plot(sr_df['σ'], sr_df['R'], 'r-s', label='Recall', linewidth=2)
ax.plot(sr_df['σ'], sr_df['F1'], 'g-^', label='F1', linewidth=2)
ax.set_xlabel('σ'); ax.set_ylabel('Skóre'); ax.set_title('Threshold analýza')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/threshold_analysis.png', dpi=150, bbox_inches='tight'); plt.show()


---
# ČASŤ 7: VIZUALIZÁCIE
---

In [ ]:
# Reálne vs Predikované – NAPÄTIE (test, Chyba=0)
# Diagonála = perfektná predikcia, body čím bližšie k nej tým lepšie
td = forecast_df
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, t in enumerate(VOLTAGE_COLS):
    ax = axes[i]; d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    r2 = r2_score(d['actual'], d['prediction'])
    # Pri väčšom počte bodov sample-neme - 5000 stačí na dobrý vizuál
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#5B4F9E')
    # Diagonálna čiara od min do max - ideálna predikcia
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4); ax.set_xlabel('Skutočné [V]'); ax.set_ylabel('Predikované [V]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')
plt.suptitle('Reálne vs Predikované – Napätie (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/scatter_voltage.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# To isté pre prúd
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, t in enumerate(CURRENT_COLS):
    ax = axes[i]; d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    r2 = r2_score(d['actual'], d['prediction'])
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#D4582A')
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4); ax.set_xlabel('Skutočné [A]'); ax.set_ylabel('Predikované [A]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')
plt.suptitle('Reálne vs Predikované – Prúdy (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/scatter_current.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Anomaly score v čase s prahom - vyberieme EIC s najviac poruchovými záznamami,
# aby sme videli, ako reziduály "vystrelia" počas reálnej poruchy
mid_h = SELECTED_HORIZONS[len(SELECTED_HORIZONS)//2]
fault_eics = anom_agg[anom_agg['Chyba']==1]['eic'].unique()
if len(fault_eics) > 0:
    # EIC s najviac poruchovými záznamami - tam bude najjasnejší "príbeh"
    best_eic = anom_agg[anom_agg['Chyba']==1].groupby('eic').size().idxmax()
    eic_data = anom_agg[(anom_agg['eic']==best_eic) & (anom_agg['hour_ahead']==mid_h)].sort_values('t_utc')
    fig, ax = plt.subplots(figsize=(14, 4))
    times = pd.to_datetime(eic_data['t_utc'])
    ax.plot(times, eic_data['anomaly_score'], color='#333', linewidth=0.6, alpha=0.8)
    # Horizontálna čiara = prah anomálie
    ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.2, label=f'Prah ({ANOMALY_SIGMA}σ)')
    # Vyfarbené pásmo = obdobia, kedy bola skutočná porucha
    fault_mask = eic_data['Chyba'].values == 1
    if fault_mask.any():
        ymax = max(eic_data['anomaly_score'].max() * 1.2, threshold * 1.3)
        ax.fill_between(times, 0, ymax, where=fault_mask, alpha=0.10, color='red', label='Skutočná porucha')
    ax.set_xlabel('Čas'); ax.set_ylabel('Anomaly score')
    ax.set_title(f'Anomaly score – EIC: {best_eic} (h{mid_h})')
    ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/anomaly_score_time.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Časový priebeh - napätie u1_norm pre náhodný EIC s dobrým R²
mid_h = SELECTED_HORIZONS[0]
td = forecast_df[(forecast_df['target'] == 'u1_norm') & 
                  (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])
WINDOW = 48

# Vyberieme len EIC, kde R² je dosť vysoké - ukážka má byť reprezentatívna
eic_r2 = {}
for eic, grp in td.groupby('eic'):
    if len(grp) >= WINDOW:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.6: eic_r2[eic] = r2

if len(eic_r2) > 0:
    np.random.seed(1)
    selected_eic = np.random.choice(list(eic_r2.keys()))
    eic_data = td[td['eic'] == selected_eic].sort_values('t_utc').reset_index(drop=True)
    # Náhodné okno z časového radu
    start_idx = np.random.randint(0, max(1, len(eic_data) - WINDOW))
    window = eic_data.iloc[start_idx:start_idx + WINDOW]
    t = pd.to_datetime(window['t_utc'])
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, window['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, window['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('U1 [V]'); ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia napätia U1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2[selected_eic]:.4f})')
    ax.legend(); ax.grid(True, alpha=0.3); ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/timeseries_voltage_forecast.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# To isté pre prúd i1_norm
td_i = forecast_df[(forecast_df['target'] == 'i1_norm') & 
                    (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])
WINDOW = 48

eic_r2_i = {}
for eic, grp in td_i.groupby('eic'):
    if len(grp) >= WINDOW:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.7: eic_r2_i[eic] = r2

if len(eic_r2_i) > 0:
    # Iný seed - aby sa nevybral ten istý EIC ako pri napätí
    np.random.seed(2)
    selected_eic = np.random.choice(list(eic_r2_i.keys()))
    eic_data = td_i[td_i['eic'] == selected_eic].sort_values('t_utc').reset_index(drop=True)
    start_idx = np.random.randint(0, max(1, len(eic_data) - WINDOW))
    window = eic_data.iloc[start_idx:start_idx + WINDOW]
    t = pd.to_datetime(window['t_utc'])
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, window['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, window['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('I1 [A]'); ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia prúdu I1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2_i[selected_eic]:.4f})')
    ax.legend(); ax.grid(True, alpha=0.3); ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/timeseries_current_forecast.png', dpi=150, bbox_inches='tight'); plt.show()


---
# ČASŤ 8: EXPORT
---

In [ ]:
# Uloženie predikcií, anomaly výsledkov a learning curve
forecast_df.to_csv(f'{OUTPUT_DIR}/tcnn_forecast_results.csv', sep=';', index=False)
anom_agg.to_csv(f'{OUTPUT_DIR}/tcnn_anomaly_results.csv', sep=';', index=False)
lc = pd.DataFrame({'epoch': range(1, len(train_losses)+1),
                    'train_loss': train_losses, 'val_loss': val_losses})
lc.to_csv(f'{OUTPUT_DIR}/tcnn_learning_curve.csv', sep=';', index=False)
print(f"Uložené do: {OUTPUT_DIR}/")
